# Fine-tuning a pre-trained model

The following project fine-tunes the Bert-base-uncased model using the yelp dataset in Hugging Face.



In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

yelp = load_dataset("yelp_polarity")

print(yelp) # To see the structure of the dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 38000
    })
})


## Tokenize the entire dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def Preprocess(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_data = yelp.map(Preprocess, batched=True)

# took 10 min 14.5 sec

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/560000 [00:00<?, ? examples/s]

Map:   0%|          | 0/38000 [00:00<?, ? examples/s]

## Tokenizing the test and train dataset

This step tokenizes the test and train datasets, trains the Bert model then save the new trained model to be used later.

In [6]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

train_yelp = yelp["train"].select(range(200))
test_yelp = yelp["test"].select(range(100))

tokenized_train = train_yelp.map(Preprocess, batched=True)
tokenized_test = test_yelp.map(Preprocess, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

model.save_pretrained("./finetuned_model")
tokenizer.save_pretrained("./finetuned_model")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Epoch,Training Loss,Validation Loss
1,No log,0.722199


Epoch,Training Loss,Validation Loss
1,No log,0.722199
2,No log,0.709119


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./finetuned_model/tokenizer_config.json', './finetuned_model/tokenizer.json')

## Evaluating the trained model

This process is concerned with checking to see how well trained is our model.

In [10]:
!pip install evaluate
import evaluate

model = AutoModelForSequenceClassification.from_pretrained("./finetuned_model")

tokenizer = AutoTokenizer.from_pretrained("./finetuned_model")

metric = evaluate.load("accuracy")

def compute_metrics(p):
    logits, labels = p
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

results = trainer.evaluate()
print("Results", results)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Results {'eval_loss': 0.7091193199157715, 'eval_model_preparation_time': 0.0052, 'eval_accuracy': 0.48, 'eval_runtime': 183.3129, 'eval_samples_per_second': 0.546, 'eval_steps_per_second': 0.071}
